# Google 2019 Cluster - Lasso Regression - `mem_usage_peak`

**Module:** cross-sectional regression &nbsp;|&nbsp; **Dataset:** Google 2019 Cluster &nbsp;|&nbsp; **Target:** `mem_usage_peak`

**Input file(s):** `borg_traces_data.csv`

**Features (request + static metadata only; target never a feature):** `cpu_request, mem_request, cpu_mem_ratio, priority, scheduling_class, collection_type`

**Split:** random 80/20, `random_state=42` &nbsp;|&nbsp; **CV:** GridSearchCV(cv=5) on TRAIN only (where applicable)

**Preprocessing:** parse dict/columns -> feature engineering -> RobustPrep(log1p + winsorise to TRAIN 1st/99th pct) -> SimpleImputer(median) -> [PolynomialFeatures(2)] -> Scaler  *(inside a Pipeline)*

**Result row -> ** `results/regression/Google_2019_Lasso_Regression_Memory_peak.csv`  (also aggregated into `results/tabular_results.csv`)

This notebook reproduces the leakage-free experiment behind `results/tabular_results.csv`. It does **not** reproduce the old target-in-features '100%' result.

In [ ]:
# Step 1: Import libraries
import ast, os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
EPS = 1e-9
RESULTS_DIR = r"../../results"

In [ ]:
# Step 15-17 helper: metric functions (MAE, MAPE, RMSE, R2, sMAPE)
# NOTE: raw MAPE is unreliable here because usage is frequently ~0 (it blows up).
#       sMAPE is the bounded percentage error actually used in the study.
def smape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.mean(2*np.abs(p - y) / (np.abs(y) + np.abs(p) + EPS)) * 100)

def all_scores(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {
        "MAE":   float(mean_absolute_error(y, p)),
        "MAPE":  float(mean_absolute_percentage_error(y, p) * 100),   # unreliable near 0
        "RMSE":  float(np.sqrt(np.mean((y - p) ** 2))),
        "R2":    float(r2_score(y, p)),
        "sMAPE": smape(y, p),
    }

class RobustPrep:
    """log1p on non-negative skewed features + winsorise to the TRAIN 1st/99th
    percentile. Bounds learned from TRAIN rows only -> no leakage."""
    def fit(self, X):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, 1, axis=0)
        self.hi_ = np.nanpercentile(X, 99, axis=0)
        self.logmask_ = np.nanmin(X, axis=0) >= 0
        return self
    def transform(self, X):
        X = np.array(X, float)
        X = np.clip(X, self.lo_, self.hi_)
        X[:, self.logmask_] = np.log1p(X[:, self.logmask_])
        return X

In [ ]:
# Step 2: Load dataset  --  Google 2019 Cluster  (Borg trace v3 public extract)
BORG_CSV = r"../../data/google_2019/borg_traces_data.csv"
raw = pd.read_csv(BORG_CSV, low_memory=False)
# reproducibility: 150,000-row random sample (random_state=42), as in the final run
raw = raw.sample(n=150_000, random_state=RANDOM_STATE)
print("raw sample:", raw.shape)

# Step 3: Data preprocessing  +  Step 4: Parse required columns
# resource_request / average_usage / maximum_usage are dict-strings -> pull cpus & memory
def dget(s, key):
    try:
        return ast.literal_eval(s).get(key, np.nan)
    except Exception:
        return np.nan

for col, pfx in [("resource_request", "req"), ("average_usage", "avg"), ("maximum_usage", "max")]:
    raw[pfx + "_cpu"] = raw[col].map(lambda s: dget(s, "cpus"))
    raw[pfx + "_mem"] = raw[col].map(lambda s: dget(s, "memory"))

# Step 5: Feature engineering  (entity table: 1 row per instance-usage record)
df = pd.DataFrame({
    "cpu_request":      pd.to_numeric(raw["req_cpu"], errors="coerce"),
    "mem_request":      pd.to_numeric(raw["req_mem"], errors="coerce"),
    "priority":         pd.to_numeric(raw["priority"], errors="coerce"),
    "scheduling_class": pd.to_numeric(raw["scheduling_class"], errors="coerce"),
    "collection_type":  pd.to_numeric(raw["collection_type"], errors="coerce"),
    "cpu_usage_mean":   pd.to_numeric(raw["avg_cpu"], errors="coerce"),
    "mem_usage_mean":   pd.to_numeric(raw["avg_mem"], errors="coerce"),
    "cpu_usage_peak":   pd.to_numeric(raw["max_cpu"], errors="coerce"),
    "mem_usage_peak":   pd.to_numeric(raw["max_mem"], errors="coerce"),
    "order_key":        pd.to_numeric(raw["start_time"], errors="coerce"),
})
df["cpu_mem_ratio"] = df["cpu_request"] / (df["mem_request"] + EPS)   # from REQUESTS -> no leak

# drop zero-request rows (ratio undefined; request=0 burst tasks are a separate regime)
df = df[(df["cpu_request"] > 0) & (df["mem_request"] > 0)]
# normalised usage lies in [0,1]; anything far outside is a parsing artefact
for c in ["cpu_usage_mean", "mem_usage_mean", "cpu_usage_peak", "mem_usage_peak"]:
    df = df[df[c].between(0, 5)]
df = df.dropna(subset=["order_key"]).sort_values("order_key").reset_index(drop=True)

FEATURES = ["cpu_request", "mem_request", "cpu_mem_ratio",
            "priority", "scheduling_class", "collection_type"]
print("entity table:", df.shape, "| features:", FEATURES)

In [ ]:
# Step 6: Define X and y      TARGET = "mem_usage_peak"
TARGET = "mem_usage_peak"
assert TARGET not in FEATURES, "target must never be a feature (this was the old 100% bug)"
X = df[FEATURES].copy()
y = df[TARGET].copy()

# Step 7: Handle missing values  -> handled inside the Pipeline by SimpleImputer(median)

# Step 8: Train/test split  --  random 80/20, random_state=42
# The regression task is CROSS-SECTIONAL (predict an unobserved property of a job
# from its request), so a shuffled split is correct and matches the original study.
Xtr_df, Xte_df, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, shuffle=True)
ytr, yte = ytr.values, yte.values
print("train:", Xtr_df.shape, " test:", Xte_df.shape)

# Step 9: Scaling / preprocessing  --  RobustPrep (log1p + winsorise to TRAIN 1st/99th pct)
# fitted on TRAIN only, then the rest (impute -> [poly] -> scale) lives in the Pipeline.
prep = RobustPrep().fit(Xtr_df.values)
Xtr = prep.transform(Xtr_df.values)
Xte = prep.transform(Xte_df.values)

In [ ]:
# Step 10: Build the model  --  Lasso (L1) on degree-2 polynomial features
# Step 11: Hyperparameter tuning  --  GridSearchCV(cv=5), alpha grid, on TRAIN only
HYPERPARAMS = {"model__alpha": [1e-4, 1e-3, 1e-2, 0.1]}
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model",  Lasso(max_iter=3000, tol=1e-3, random_state=RANDOM_STATE)),
])
gs = GridSearchCV(pipe, HYPERPARAMS, cv=5, scoring="neg_mean_absolute_error", n_jobs=4)
gs.fit(Xtr, ytr)
best = gs.best_estimator_
print("best params:", gs.best_params_)
HYPERPARAMS = gs.best_params_

# Step 12-14
train_pred = best.predict(Xtr)
test_pred  = best.predict(Xte)

In [ ]:
# Step 18: Plot actual vs predicted (test set)
plt.figure(figsize=(6, 6))
plt.scatter(yte, test_pred, s=8, alpha=0.35)
lim = [min(yte.min(), test_pred.min()), max(yte.max(), test_pred.max())]
plt.plot(lim, lim, "r--", lw=1)
plt.xlabel("Actual " + TARGET); plt.ylabel("Predicted " + TARGET)
plt.title("Google 2019 Cluster - Lasso Regression - " + TARGET + " (test)")
plt.tight_layout(); plt.show()

# Step 19: Print / save final results
train_s = all_scores(ytr, train_pred)
test_s  = all_scores(yte, test_pred)
print("\nGoogle 2019 Cluster  |  Lasso Regression  |  " + TARGET)
print("  TRAIN  MAE {:.4f}  MAPE {:.1f}%  RMSE {:.4f}  R2 {:.4f}  sMAPE {:.1f}".format(
    train_s["MAE"], train_s["MAPE"], train_s["RMSE"], train_s["R2"], train_s["sMAPE"]))
print("  TEST   MAE {:.4f}  MAPE {:.1f}%  RMSE {:.4f}  R2 {:.4f}  sMAPE {:.1f}".format(
    test_s["MAE"], test_s["MAPE"], test_s["RMSE"], test_s["R2"], test_s["sMAPE"]))

row = {"dataset": "Google 2019 Cluster", "module": "regression", "target": TARGET, "model": "Lasso Regression",
       "n_train": len(ytr), "n_test": len(yte),
       "train_MAE": train_s["MAE"], "train_MAPE": train_s["MAPE"], "train_RMSE": train_s["RMSE"],
       "train_R2": train_s["R2"], "train_sMAPE": train_s["sMAPE"],
       "test_MAE": test_s["MAE"], "test_MAPE": test_s["MAPE"], "test_RMSE": test_s["RMSE"],
       "test_R2": test_s["R2"], "test_sMAPE": test_s["sMAPE"],
       "hyperparams": json.dumps(HYPERPARAMS)}
os.makedirs(os.path.join(RESULTS_DIR, "regression"), exist_ok=True)
out_csv = os.path.join(RESULTS_DIR, "regression", "Google_2019_Lasso_Regression_Memory_peak.csv")
pd.DataFrame([row]).to_csv(out_csv, index=False)
print("\nsaved ->", out_csv)